In [ ]:
#License: MIT

#https://github.com/gduscher/MLSTEM2025/blob/main/Day%201/5_Atom_Finding.ipynb

%matplotlib widget

import matplotlib.pylab as plt
import numpy as np
import os
import sys

#%load_ext autoreload
#%autoreload 2    
#sys.path.insert(0, '../sidpy')
import sidpy
#sys.path.insert(0, '../pyTEMlib')
import pyTEMlib

#sys.path.insert(0, '../../pyTEMlib')
import pyTEMlib.file_tools      # File input/ output library
import pyTEMlib.image_tools 
import pyTEMlib.probe_tools
import pyTEMlib.atom_tools

# For archiving reasons it is a good idea to print the version numbers out at this point
print('pyTEM version: ', pyTEMlib.__version__)


import cv2

import hyperspy.api as hs
import atomap.api as am
import skimage

In [ ]:
folder='/path/to/folder/'

In [ ]:
#fileWidget.file_name
fname = '/path/fname.emd'



datasets = pyTEMlib.file_tools.open_file(fname)
#datasets = pyTEMlib.file_tools.open_file(fileWidget.file_name)#
selector = sidpy.ChooseDataset(datasets)



In [ ]:
selector.dataset

In [ ]:


dataset = selector.dataset
dataset.data_type = 'IMAGE_STACK'
dataset.x.dimension_type = 'SPATIAL'
dataset.y.dimension_type = 'SPATIAL'

view = dataset.plot()
dataset.x.slope



In [ ]:
dataset.shape

In [ ]:
#lets crop it to the fruitful area only

#dataset = dataset[:31]
dataset = dataset[:16]

In [ ]:
'''
Here we use the Diffeomorphic Demon Non-Rigid Registration as provided by simpleITK.
Please Cite:
    simpleITK
    and
    T. Vercauteren, X. Pennec, A. Perchant and N. Ayache Diffeomorphic Demons Using ITK's Finite Difference Solver Hierarchy The Insight Journal, 2007
'''

rigid_registered= pyTEMlib.image_tools.rigid_registration(dataset)
datasets['rigid_registered'] = rigid_registered
view = rigid_registered.plot()



In [ ]:
demon_registered = pyTEMlib.image_tools.demon_registration(rigid_registered)
datasets['demon_registered'] = demon_registered
view2 = demon_registered.plot()


In [ ]:
plt.close('all')
drift = rigid_registered.metadata['analysis']['rigid_registration']['drift']
polynom_degree = 2 # 1 is linear fit, 2 is parabolic fit, ...

x = np.linspace(0,drift.shape[0]-1,drift.shape[0])

line_fit_x = np.polyfit(x, drift[:,0], polynom_degree)
poly_x = np.poly1d(line_fit_x)
line_fit_y = np.polyfit(x, drift[:,1], polynom_degree)
poly_y = np.poly1d(line_fit_y)

plt.figure()
plt.axhline(color = 'gray')
plt.plot(x, drift[:,0], label = 'drift x')
plt.plot(x, drift[:,1], label = 'drift y')
plt.plot(x, poly_x(x),  label = 'fit_drift_x')
plt.plot(x, poly_y(x),  label = 'fit_drift_y')

plt.legend();
ax_pixels = plt.gca()
ax_pixels.step(1, 1)

scaleX = 1000 #(rigid_registered.x[1]-rigid_registered.x[0])*1000.  #in pm

ax_pm = ax_pixels.twinx()
x_1, x_2 = ax_pixels.get_ylim()

ax_pm.set_ylim(x_1*scaleX, x_2*scaleX)

ax_pixels.set_ylabel('drift [pixels]')
ax_pm.set_ylabel('drift [pm]')
ax_pixels.set_xlabel('image number');
plt.tight_layout()

In [ ]:
rigid_registered.metadata

In [ ]:
plt.close('all')
#plt.imshow(np.mean(demon_registered,axis=0)[75:325,50:1475])
plt.imshow(dataset)
plt.show()

In [ ]:
cv2.imwrite(folder+fname+'.tif', np.mean(demon_registered,axis=0).compute())

In [ ]:
s = hs.load(fname)

In [ ]:
s[0].original_metadata.BinaryResult.PixelSize

In [ ]:
# ------- Input ------
#atoms_size = .065 # in nm
atoms_size = .075 # in nm
# --------------------

image = demon_registered.sum(axis=0)

out_tags = {}
#image.metadata['experiment']= {'convergence_angle': 30, 'acceleration_voltage': 200000.}

In [ ]:
#https://github.com/gduscher/MLSTEM2025/blob/main/Day%201/5_Atom_Finding.ipynb



scale_x = image.x.slope
gauss_diameter = atoms_size/scale_x
gauss_probe = pyTEMlib.probe_tools.make_gauss(image.shape[0], image.shape[1], gauss_diameter)

print('Deconvolution of ', dataset.title)
LR_dataset = pyTEMlib.image_tools.decon_lr(image, gauss_probe, verbose=False)
datasets['Log_001'] = LR_dataset
extent = LR_dataset.get_extent([0,1])
fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
ax[0].imshow(image.T, extent = extent,vmax=np.median(np.array(image))+3*np.std(np.array(image)))
ax[1].imshow(LR_dataset.T, extent = extent, vmax=np.median(np.array(LR_dataset))+3*np.std(np.array(LR_dataset)));


In [ ]:

# ------- Input ------
threshold = 0.9 #usally between 0.01 and 0.9  the smaller the more atoms
atom_size = .06 #in nm  
# ----------------------


image = LR_dataset
#image = image_choice.dataset
scale_x = image.x.slope
blobs =  skimage.feature.blob_log(image, max_sigma=atom_size/scale_x, threshold=threshold)

fig1, ax = plt.subplots(1, 1,figsize=(8,7), sharex=True, sharey=True)
plt.title("blob detection ")

plt.imshow(image.T, interpolation='nearest',cmap='gray', vmax=np.median(np.array(image))+3*np.std(np.array(image)))
plt.scatter(blobs[:, 0], blobs[:, 1], c='r', s=20, alpha = .5);


In [ ]:
atoms = blobs
image = demon_registered.sum(axis=0)
image = image-image.min()

atom_radius = 2
MaxInt = 0
MinInt = 0
maxDist = 2
sym = pyTEMlib.atom_tools.atom_refine(np.array(image), atoms, atom_radius, max_int = 0, min_int = 0, max_dist = 2)
refined_atoms = np.array(sym['atoms'])

fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
ax[0].imshow(image.T)
ax[0].scatter(refined_atoms[:,0],refined_atoms[:,1],  s=10, alpha = 0.3, color = 'red')
ax[0].set_title('refined atom postion')
ax[1].imshow(image.T)
ax[1].scatter(atoms[:, 0], atoms[:, 1], c='r', s=10, alpha = .3);
ax[1].set_title('blobs on image');


In [ ]:
plt.close('all')
plt.violinplot(sym['gauss_intensity'])
plt.show()

In [ ]:
#Export

#x = sublattice.x_position
#y = sublattice.y_position
#size = sublattice.pixel_size
#ellipticity = np.asarray(sublattice.ellipticity) - 1
#rot = -np.asarray(sublattice.rotation_ellipticity)
#np.save(folder+'planar_928_pyTEMlib',np.array(sym['atoms']))

In [ ]:

#import atomap

In [ ]:
#atom_positions = [atomap.atom_position.Atom_Position(x, y) for x, y in sym['atoms']]
sublattice = am.Sublattice(sym['atoms'], image=image.T)
sublattice.find_nearest_neighbors()

In [ ]:
sublattice.refine_atom_positions_using_2d_gaussian(
    percent_to_nn=0.4,   # default; fraction of NN distance used as fit mask
)

In [ ]:
sublattice.plot()

In [ ]:
i_points, i_record, p_record = sublattice.integrate_column_intensity()

In [ ]:
x = sublattice.x_position
y = sublattice.y_position
#size = sublattice.pixel_size
ellipticity = np.asarray(sublattice.ellipticity) - 1
rot = -np.asarray(sublattice.rotation_ellipticity)
#np.save(folder+'planar_928_pyTEMlib_atomap',np.array([x,y,ellipticity,rot,i_points]))
np.save(folder+fname,np.array([x,y,ellipticity,rot,i_points]))

In [ ]:
fft_image = dataset.fft().abs()
fft_image.plot()

In [ ]:
dataset.shape